In [ ]:
# celda 0
# cuadernillo A para evaluar scope y reacciones de semillas triviales y cofactores
# se filtran reacciones problematicas (cuadernillo B las elimina y recalcula scope)
# en total demora como 30 minutos en correr
# se generan archivos JSON con reacciones madre que inflan el scope con la semilla de cofs

# montar Drive
from google.colab import drive
drive.mount('/content/drive')

# importar paquetes
import subprocess
subprocess.run(["pip", "install", "-q", "python-libsbml", "menetools", "clyngor-with-clingo"])

import os
import libsbml
import pandas as pd
from menetools import run_menescope, run_meneacti

## funciones generales

# leer modelo sbml sin cobrapy
def load_model(file_path):
    reader = libsbml.SBMLReader()
    doc = reader.readSBML(file_path)
    if doc.getNumErrors() > 0:
        print(f"Error leyendo {file_path}:")
        doc.printErrors()
        return None
    return doc.getModel()

# encontrar el modelo sbml en las carpetas
def buscar_sbml(carpeta):
    for f in os.listdir(carpeta):
        if f.endswith('.sbml'):
            return os.path.join(carpeta, f)
    raise FileNotFoundError(carpeta)

# leer reaccion
def reaction_to_string(reaction, model):
    """Devuelve la reacción como string estequiométrico."""
    def species_str(spec_ref):
        sp = model.getSpecies(spec_ref.getSpecies())
        coeff = spec_ref.getStoichiometry()
        coeff_str = "" if coeff == 1 else str(coeff) + " "
        return coeff_str + (sp.getId() if sp is not None else spec_ref.getSpecies())

    reactants = " + ".join([species_str(r) for r in reaction.getListOfReactants()])
    products  = " + ".join([species_str(p) for p in reaction.getListOfProducts()])
    return f"{reaction.getId()}: {reactants} -> {products}"

Mounted at /content/drive


In [ ]:
# celda 1
# modificar RUTAS si necesario

MODELOS = '/content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2'
SITIOS = ['Las_Docas', 'Algarrobo', 'Navidad', 'Topocalma', 'Ilque', 'San_Antonio', 'Pargua', 'Los_Chonos']

# funcion "buscar_sbml" toma el .sbml que haya en la carpeta, sin importar como se llame el archivo

# modelos sbml v1 (curacion con los 30 pares de cofactors.tsv)
rutas_v1 = {sitio: buscar_sbml(os.path.join(MODELOS, sitio, 'sbml_curado')) for sitio in SITIOS}

# carpeta test: "/content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test"
CARPETA_TEST = os.path.join(MODELOS, 'COFACTORES', 'test')

# semillas triviales
SEED_WATER  = os.path.join(CARPETA_TEST, 'seed_OnlyWater.sbml')
SEED_AMP    = os.path.join(CARPETA_TEST, 'seed_OnlyAMP.sbml')
SEED_PPI    = os.path.join(CARPETA_TEST, 'seed_OnlyPPI.sbml')
SEED_PROTON = os.path.join(CARPETA_TEST, 'seed_OnlyProton.sbml')
SEED_NOTHING = os.path.join(CARPETA_TEST, 'seed_OnlyNothing.sbml')

print("Rutas v1:", {k: os.path.basename(v) for k, v in rutas_v1.items()})
print("Semillas triviales:")
for nombre, ruta in [('agua', SEED_WATER), ('amp', SEED_AMP), ('ppi', SEED_PPI), ('proton', SEED_PROTON)]:
    print(f"  {nombre}: {ruta}  (existe: {os.path.exists(ruta)})")

Rutas v1: {'Las_Docas': 'ld_metagenome_withgenes.sbml', 'Algarrobo': 'al_metagenome_withgenes.sbml', 'Navidad': 'nav_metagenome_withgenes.sbml', 'Topocalma': 'top_metagenome_withgenes.sbml', 'Ilque': 'ilq_metagenome_withgenes.sbml', 'San_Antonio': 'sant_metagenome_withgenes.sbml', 'Pargua': 'par_metagenome_withgenes.sbml', 'Los_Chonos': 'lc_metagenome_withgenes.sbml'}
Semillas triviales:
  agua: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyWater.sbml  (existe: True)
  amp: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyAMP.sbml  (existe: True)
  ppi: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyPPI.sbml  (existe: True)
  proton: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyProton.sbml  (existe: True)


In [ ]:
# celda 2
# funcion para obtener scope y reacciones activadas segun semilla trivial
# imprime listado y tabla resumen
# se toma sus minutos

def analizar_semilla_trivial(nombre_semilla, seed_path, rutas, sitios=SITIOS):
    """
    Corre menescope y meneacti para una semilla trivial sobre todos los sitios.
    Imprime por sitio: scope completo, y reacciones activadas con estequiometria.
    Devuelve (df_resumen, detalle) donde detalle[sitio] = {'scope':..., 'reacciones_activadas':...}
    """
    resumen = []
    detalle = {}

    for site in sitios:
        sbml_path = rutas[site]

        resultado_scope = run_menescope(draft_sbml=sbml_path, seeds_sbml=seed_path) #menescope
        scope = sorted(resultado_scope['scope'])

        activadas = run_meneacti(draft_sbml=sbml_path, seeds_sbml=seed_path) #meneacti
        model = load_model(sbml_path)

        print("=" * 70)
        print(f"SITIO: {site}   |   SEMILLA: {nombre_semilla}")
        print("=" * 70)

        print(f"\n--- Metabolitos en el scope ({len(scope)}) ---")
        for m in scope:
            print(" ", m)

        print(f"\n--- Reacciones activadas ({len(activadas)}) ---")
        for rid in activadas:
            reaction = model.getReaction(rid) if model else None
            if reaction is None:
                print(f"   No se encontró la reacción {rid}")
            else:
                print("  ", reaction_to_string(reaction, model))

        resumen.append({
            'sitio': site,
            'semilla': nombre_semilla,
            'n_metabolitos_scope': len(scope),
            'n_reacciones_activadas': len(activadas)
        })
        detalle[site] = {'scope': scope, 'reacciones_activadas': activadas}
        print()

    df_resumen = pd.DataFrame(resumen)
    print("=" * 70)
    print(f"TABLA RESUMEN — semilla '{nombre_semilla}'")
    print("=" * 70)
    print(df_resumen)

    return df_resumen, detalle

In [ ]:
# celda 3
df_proton, detalle_proton = analizar_semilla_trivial('proton', SEED_PROTON, rutas_v1)

SITIO: Las_Docas   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: Algarrobo   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: Navidad   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: Topocalma   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: Ilque   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: San_Antonio   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: Pargua   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

SITIO: Los_Chonos   |   SEMILLA: proton

--- Metabolitos en el scope (1) ---
  M_PROTON_c

--- Reacciones activadas (0) ---

TABLA RESUMEN 

In [ ]:
# celda 4
df_water, detalle_water = analizar_semilla_trivial('agua', SEED_WATER, rutas_v1)
# OBS: se obtienen 2 metabolitos pues la reaccion activada es reversible


SITIO: Las_Docas   |   SEMILLA: agua

--- Metabolitos en el scope (2) ---
  M_WATER_c
  M_WATER_e

--- Reacciones activadas (1) ---
   R_TRANS__45__RXN__45__145: M_WATER_e -> M_WATER_c

SITIO: Algarrobo   |   SEMILLA: agua

--- Metabolitos en el scope (2) ---
  M_WATER_c
  M_WATER_e

--- Reacciones activadas (1) ---
   R_TRANS__45__RXN__45__145: M_WATER_e -> M_WATER_c

SITIO: Navidad   |   SEMILLA: agua

--- Metabolitos en el scope (2) ---
  M_WATER_c
  M_WATER_e

--- Reacciones activadas (1) ---
   R_TRANS__45__RXN__45__145: M_WATER_e -> M_WATER_c

SITIO: Topocalma   |   SEMILLA: agua

--- Metabolitos en el scope (2) ---
  M_WATER_c
  M_WATER_e

--- Reacciones activadas (1) ---
   R_TRANS__45__RXN__45__145: M_WATER_e -> M_WATER_c

SITIO: Ilque   |   SEMILLA: agua

--- Metabolitos en el scope (2) ---
  M_WATER_c
  M_WATER_e

--- Reacciones activadas (1) ---
   R_TRANS__45__RXN__45__145: M_WATER_e -> M_WATER_c

SITIO: San_Antonio   |   SEMILLA: agua

--- Metabolitos en el scope (2) ---


In [ ]:
# celda 5
df_amp, detalle_amp = analizar_semilla_trivial('amp', SEED_AMP, rutas_v1)

SITIO: Las_Docas   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: Algarrobo   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: Navidad   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: Topocalma   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: Ilque   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: San_Antonio   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: Pargua   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

SITIO: Los_Chonos   |   SEMILLA: amp

--- Metabolitos en el scope (1) ---
  M_AMP_c__cof__

--- Reacciones activadas (0) ---

TABLA 

In [ ]:
# celda 6
df_ppi, detalle_ppi = analizar_semilla_trivial('ppi', SEED_PPI, rutas_v1)

SITIO: Las_Docas   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: Algarrobo   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: Navidad   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: Topocalma   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: Ilque   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: San_Antonio   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: Pargua   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

SITIO: Los_Chonos   |   SEMILLA: ppi

--- Metabolitos en el scope (1) ---
  M_PPI_c

--- Reacciones activadas (0) ---

TABLA RESUMEN — semilla 'ppi'
         sitio semilla  n_metabo

In [ ]:
# celda 6.5
df_nothing, detalle_nothing = analizar_semilla_trivial('nothing', SEED_NOTHING, rutas_v1)


SITIO: Las_Docas   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: Algarrobo   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: Navidad   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: Topocalma   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: Ilque   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: San_Antonio   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: Pargua   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

SITIO: Los_Chonos   |   SEMILLA: nothing

--- Metabolitos en el scope (0) ---

--- Reacciones activadas (0) ---

TABLA RESUMEN — semilla 'nothing'
         sitio  semilla  n_metabolitos_scope  n_reacciones_activadas
0    La

In [ ]:
# celda 7

# leer metabolitos del SBML de cofactores
# util para encontrar las reacciones "madre" de metabolitos fuga
def cargar_especies_semilla(seed_path):
    """Devuelve el set de IDs de especies presentes en un SBML de semilla."""
    doc = libsbml.SBMLReader().readSBML(seed_path)
    m = doc.getModel()
    return set(s.getId() for s in m.getListOfSpecies())

# para evitar listados muy gigantes
def imprimir_lista_truncada(items, limite=100):
    """Imprime una lista completa, o truncada con aviso si supera 'limite' elementos."""
    if len(items) <= limite:
        for m in items:
            print(" ", m)
    else:
        for m in items[:limite]:
            print(" ", m)
        print(f"   ... ({len(items) - limite} metabolitos más, omitidos por espacio)")


In [ ]:
# celda 8

# funcion para analizar scope y reacciones de semilla cofactores
def analizar_semilla_grande(nombre_semilla, seed_path, rutas, sitios=SITIOS, limite_impresion=100):
    """
    Analiza una semilla grande (ej. cofactores) usando run_menescope + run_meneacti
    (ASP/clingo). Imprime por sitio: scope, fuga (no-cof) y reacciones responsables;
    devuelve (df_resumen, detalle).
    """
    resumen = []
    detalle = {}

    for site in sitios:
        sbml_path = rutas[site]

        resultado_scope = run_menescope(draft_sbml=sbml_path, seeds_sbml=seed_path)
        scope = sorted(resultado_scope['scope'])
        activadas = run_meneacti(draft_sbml=sbml_path, seeds_sbml=seed_path)
        fuga = sorted([m for m in scope if '__cof__' not in m])

        model = load_model(sbml_path)
        reacciones_fuga = []
        for r in model.getListOfReactions():
            rid = r.getId()
            if rid not in activadas:
                continue
            products_no_cof = [p.getSpecies() for p in r.getListOfProducts() if '__cof__' not in p.getSpecies()]
            if products_no_cof:
                reacciones_fuga.append(rid)

        print("=" * 70)
        print(f"SITIO: {site}   |   SEMILLA: {nombre_semilla}")
        print("=" * 70)
        print(f"\n--- Metabolitos en el scope ({len(scope)}) ---")
        imprimir_lista_truncada(scope, limite_impresion)
        print(f"\n--- Metabolitos FUGA, no-cof ({len(fuga)}) ---")
        imprimir_lista_truncada(fuga, limite_impresion)
        print(f"\n--- Reacciones responsables de la fuga ({len(reacciones_fuga)}) ---")
        for rid in reacciones_fuga:
            reaction = model.getReaction(rid)
            print("  ", reaction_to_string(reaction, model))

        resumen.append({'sitio': site, 'semilla': nombre_semilla,
                         'n_metabolitos_scope': len(scope), 'n_reacciones_activadas': len(activadas),
                         'n_metabolitos_fuga': len(fuga), 'n_reacciones_fuga': len(reacciones_fuga)})
        detalle[site] = {'scope': scope, 'reacciones_activadas': activadas,
                          'fuga': fuga, 'reacciones_fuga': reacciones_fuga}
        print()

    df_resumen = pd.DataFrame(resumen)
    print("=" * 70)
    print(f"TABLA RESUMEN — semilla '{nombre_semilla}'")
    print("=" * 70)
    print(df_resumen)
    return df_resumen, detalle

In [ ]:
# celda 9

# semilla cofactores
SEED_COF_V1 = os.path.join(MODELOS, 'COFACTORES', 'seed_cofactors.sbml')

# calcular scope y reacciones con semilla de cofactores
df_cof_v1, detalle_cof_v1 = analizar_semilla_grande('cofactores_v1', SEED_COF_V1, rutas_v1)


Se truncaron las últimas líneas 5000 del resultado de transmisión.
   R_2__46__3__46__1__46__157__45__RXN: M_ACETYL__45__COA_c + M_GLUCOSAMINE__45__1P_c -> M_N__45__ACETYL__45__D__45__GLUCOSAMINE__45__1__45__P_c + M_CO__45__A_c + M_PROTON_c
   R_2__46__4__46__1__46__123__45__RXN: M_MYO__45__INOSITOL_c + M_CPD__45__14553_c -> M_UDP_c + M_CPD__45__458_c + M_PROTON_c
   R_2__46__4__46__1__46__124__45__RXN: M_BETA__45__D__45__GALACTOSYL__45__ETCETERA__45__GLUCOSAMINE_c + M_CPD__45__14553_c -> M_PROTON_c + M_UDP_c + M_ALPHA__45__D__45__GALACTOSYL__45__13__45__BETA__45__D__45__GALACTOS_c
   R_2__46__4__46__1__46__82__45__RXN: M_SUCROSE_c + M_CPD__45__458_c -> M_CPD__45__1099_c + M_MYO__45__INOSITOL_c
   R_2__46__5__46__1__46__19__45__RXN: M_SHIKIMATE__45__5P_c + M_PHOSPHO__45__ENOL__45__PYRUVATE_c -> M_3__45__ENOLPYRUVYL__45__SHIKIMATE__45__5P_c + M_Pi_c
   R_2__46__5__46__1__46__44__45__RXN: 2.0 M_PUTRESCINE_c -> M_AMMONIUM_c + M_CPD__45__1821_c
   R_2__46__5__46__1__46__45__45__RXN: M_PUTR

In [ ]:
# celda 10

# filtrar y encontrar las reacciones "madre" de todos los metabolitos fuga
def clasificar_reacciones_fuga(model, seed_species, max_iter=50):
    """
    Expande el scope trackeando el ORIGEN de cada reaccion de fuga:
    - 'madre': todos sus reactivos estan en la semilla original -> causa raiz
    - 'derivada': al menos un reactivo es un metabolito ya filtrado en una
                  iteracion anterior -> fuga en cascada, depende de una madre
    Devuelve tres listas: madres, derivadas, y el detalle con iteracion de disparo.
    """
    disponible = set(seed_species)
    activas_ids = set()
    detalle_fuga = []  # (rid, iteracion, tipo, productos_no_cof)

    rxns_info = []
    for r in model.getListOfReactions():
        reactants = [sp.getSpecies() for sp in r.getListOfReactants()]
        products  = [sp.getSpecies() for sp in r.getListOfProducts()]
        rxns_info.append((r.getId(), reactants, products, r.getReversible()))

    for it in range(max_iter):
        nuevas = False
        for rid, reactants, products, reversible in rxns_info:
            if rid in activas_ids:
                continue

            disparo = None
            if reactants and all(sp in disponible for sp in reactants):
                disparo = reactants
            elif reversible and products and all(sp in disponible for sp in products):
                disparo = products
                reactants, products = products, reactants  # sentido inverso

            if disparo is None:
                continue

            activas_ids.add(rid)
            disponible.update(products)
            nuevas = True

            productos_no_cof = [p for p in products if '__cof__' not in p]
            if productos_no_cof:
                tipo = 'madre' if all(sp in seed_species for sp in disparo) else 'derivada'
                detalle_fuga.append((rid, it, tipo, productos_no_cof))

        if not nuevas:
            break

    madres    = [d for d in detalle_fuga if d[2] == 'madre']
    derivadas = [d for d in detalle_fuga if d[2] == 'derivada']
    return madres, derivadas, detalle_fuga

In [ ]:
# celda 11

# para liberar RAM:
import gc

resumen_madre = []
reacciones_madre_por_sitio = {}
reacciones_derivadas_por_sitio = {}
estequiometria_madre = {}   # rid -> string de la reaccion (se guarda una sola vez)
sitios_por_madre = {}       # rid -> lista de sitios donde aparece como madre

for site in SITIOS:
    sbml_path = rutas_v1[site]
    model = load_model(sbml_path)
    seed_species = cargar_especies_semilla(SEED_COF_V1)

    madres, derivadas, _ = clasificar_reacciones_fuga(model, seed_species)

    print("=" * 70)
    print(f"SITIO: {site}  |  madres: {len(madres)}  |  derivadas: {len(derivadas)}")
    print("=" * 70)

    print(f"\n--- Reacciones MADRE ({len(madres)}) ---")
    for rid, it, tipo, prods in madres:
        reaction = model.getReaction(rid)
        texto = reaction_to_string(reaction, model)
        print("  ", texto)
        print(f"     -> genera de fuga: {prods}")

        if rid not in estequiometria_madre:      # guarda el texto solo la primera vez que aparece
            estequiometria_madre[rid] = texto
        sitios_por_madre.setdefault(rid, []).append(site)
    print()

    resumen_madre.append({'sitio': site, 'n_madres': len(madres), 'n_derivadas': len(derivadas)})
    reacciones_madre_por_sitio[site] = [d[0] for d in madres]
    reacciones_derivadas_por_sitio[site] = [d[0] for d in derivadas]

    del model
    gc.collect()

df_madre = pd.DataFrame(resumen_madre)
print("=" * 70)
print("TABLA RESUMEN")
print("=" * 70)
print(df_madre)

# ============================================================
# LISTADO CONSOLIDADO — reacciones madre únicas (sin repetir por sitio)
# ============================================================
print("\n" + "=" * 70)
print(f"LISTADO CONSOLIDADO DE REACCIONES MADRE ({len(estequiometria_madre)} distintas)")
print("=" * 70)
for rid in sorted(estequiometria_madre):
    print("  ", estequiometria_madre[rid])
    print(f"     presente en: {sitios_por_madre[rid]}")

SITIO: Las_Docas  |  madres: 6  |  derivadas: 1353

--- Reacciones MADRE (6) ---
   R_1__46__18__46__1__46__2__45__RXN__cof__: M_NADP_e__cof__ + 2.0 M_Reduced__45__ferredoxins_e__cof__ + M_PROTON_e -> M_NADPH_e__cof__ + 2.0 M_Oxidized__45__ferredoxins_e__cof__
     -> genera de fuga: ['M_PROTON_e']
   R_ACETYL__45__COA__45__ACETYLTRANSFER__45__RXN__cof__: 2.0 M_ACETYL__45__COA_c__cof__ -> M_ACETOACETYL__45__COA_c + M_CO__45__A_c__cof__
     -> genera de fuga: ['M_ACETOACETYL__45__COA_c']
   R_ADENYL__45__KIN__45__RXN__cof__: M_ATP_c__cof__ + M_AMP_c -> 2.0 M_ADP_c__cof__
     -> genera de fuga: ['M_AMP_c']
   R_NADPH__45__DEHYDROGENASE__45__RXN__cof__: M_Acceptor_c__cof__ + M_NADPH_c__cof__ + M_PROTON_c -> M_Donor__45__H2_c__cof__ + M_NADP_c__cof__
     -> genera de fuga: ['M_PROTON_c']
   R_RXN__45__12444__cof__: M_FMNH2_c__cof__ + M_NADP_c__cof__ -> M_FMN_c__cof__ + M_NADPH_c__cof__ + 2.0 M_PROTON_c
     -> genera de fuga: ['M_PROTON_c']
   R_RXN0__45__4141__cof__: M_HYDROGEN__45__MO

In [ ]:
# celda 12

import json
# guardar en Drive reusltados de reacciones madre por sitio (formato JSON)
# asi evito correr todo de nuevo para cuadernillo B

ruta_checkpoint = os.path.join(MODELOS, 'COFACTORES', 'reacciones_madre_por_sitio.json')
with open(ruta_checkpoint, 'w') as f:
    json.dump(reacciones_madre_por_sitio, f, indent=2)

print("Checkpoint guardado en:", ruta_checkpoint)
print("Reacciones madre por sitio:", {s: len(v) for s, v in reacciones_madre_por_sitio.items()})

Checkpoint guardado en: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/reacciones_madre_por_sitio.json
Reacciones madre por sitio: {'Las_Docas': 6, 'Algarrobo': 7, 'Navidad': 6, 'Topocalma': 7, 'Ilque': 5, 'San_Antonio': 7, 'Pargua': 7, 'Los_Chonos': 7}
